# 04e · Ablations (C6)

**Purpose.** R1: T' at half and double the default. R2: steady state features (1 to 4) vs all 12. R3: 16 vs 32 vs 64 features per sensor.

**Inputs**
- Both `X_signals.npy` + `meta.csv`

**Outputs**
- R1 GWT (600, 3600) / (600, 14400); ES (144, 385) / (144, 1540)
- R2 GWT (600, 288) vs (600, 864); ES (144, 44) vs (144, 132)
- R3 GWT (600, 1152) / (600, 2304) / (600, 4608); ES (144, 176) / (144, 352) / (144, 704)
- `results/04e_ablations.csv`

In [ ]:
import sys
sys.path.append("../..")          # project root, so that `import src` works

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("../..")
from src.cv import load_baseline_level, load_processed, precomputed, run_cv, save_results
from src.features.encoder import encoder_builder
from src.features.handcrafted import STEADY_STATE, handcrafted_features
from src.features.raw import raw_features

NAME = "04e_ablations"
TABLES = ROOT / "report/tables"
TASKS_ABL = ["multi"]       # add "binary" if time allows (R3 ablations are the slow part)
T_PRIME = {"gwt": 100, "es": 70}

## C6: small ablations (the defaults come from the main results)
| Representation | Variants run here | Default (main results) |
|---|---|---|
| R1 | T' at half and double the default | T' = 100 (GWT), 70 (ES) |
| R2 | steady state features 1 to 4 only | all 12 |
| R3 | 16 and 64 features per sensor | 32 |

In [ ]:
parts = []
for dataset in ["gwt", "es"]:
    X, meta, info = load_processed(dataset)                       # (N, C, T)
    levels = load_baseline_level(dataset)                         # (N, C)
    t = T_PRIME[dataset]
    runs = [
        (f"R1_raw_T{t // 2}", raw_features(X, t // 2), precomputed),            # (N, C * T'/2)
        (f"R1_raw_T{t * 2}", raw_features(X, t * 2), precomputed),              # (N, C * 2T')
        ("R2_steady_state_1to4", handcrafted_features(X, levels, info["n_base"], info["exposure_end"],
                                                      subset=STEADY_STATE), precomputed),   # (N, C * 4)
        ("R3_encoder_16", X, encoder_builder(n_out=16)),                       # (N, C * 16)
        ("R3_encoder_64", X, encoder_builder(n_out=64)),                       # (N, C * 64)
    ]
    for task in TASKS_ABL:
        for rep, data, builder in runs:
            s, _ = run_cv(data, meta, dataset, task, rep, builder, verbose=False)
            parts.append(s)
            print(dataset, task, rep, round(s.groupby("head")["macro_f1"].mean().max(), 3))
scores = pd.concat(parts, ignore_index=True)
save_results(scores, None, NAME)

In [ ]:
main = pd.concat([pd.read_csv(ROOT / f"results/{n}.csv") for n in
                  ["01a_gwt_raw", "01b_es_raw", "02b_gwt_handcrafted", "02c_es_handcrafted", "03a_gwt_encoder", "03b_es_encoder"]])
main = main[main["task"].isin(TASKS_ABL)].replace({"rep": {"R1_raw": "R1_raw (default)", "R2_handcrafted": "R2 all 12 (default)",
                                                           "R3_encoder": "R3_encoder_32 (default)"}})
allv = pd.concat([main, scores], ignore_index=True)
table = (allv.groupby(["dataset", "task", "rep", "head"])["macro_f1"].mean().round(3)
         .unstack("head")[["logreg", "svm", "rf", "mlp"]])
table.to_csv(TABLES / "C6_ablations.csv")
table